# **Prediksi Praktikum 08 - IF5100 Pemrograman untuk Data Analitik**

Nama : ______________________

NIM  : ______________________

### Aturan / Rules

- Kerjakan di **Google Colab** (runtime CPU). Jalankan sel **berurutan** dari atas ke bawah, karena tiap soal memakai hasil soal sebelumnya.
- Library yang boleh dipakai: `numpy`, `pandas`, `matplotlib`, `seaborn`.
- **Jangan ubah** nama fungsi, parameter, dan kode uji. Output grafik tidak harus 100% sama, tetapi nilai yang dikembalikan fungsi harus benar.
- Setiap grafik wajib punya **judul, label sumbu, dan legend** (jika ada lebih dari satu seri).

## Studi kasus: Kopi Senja

Kamu adalah analis data di jaringan kedai **Kopi Senja** yang punya 4 cabang: Bandung, Jakarta, Surabaya, dan Yogyakarta.
Manajer memberikan data transaksi **September–Oktober 2026**. Data ini diambil dari 4 sistem kasir yang berbeda,
sehingga **kotor**: penulisan nama cabang tidak seragam, harga tersimpan sebagai teks (`"Rp 27.000"`, `"?"`),
ada umur pelanggan yang mustahil, rating kosong, transaksi ganda, dan kolom yang isinya sama semua.

Tugasmu: **membersihkan data**, lalu menjawab pertanyaan bisnis manajer dengan **visualisasi** yang tepat.

*You are a data analyst at the Kopi Senja coffee chain (4 branches). The September–October 2026 transaction data is messy. Clean it, then answer the manager's business questions with the right visualizations.*

| Kolom | Isi | Tipe |
|---|---|---|
| `id_transaksi` | ID unik transaksi | identifier |
| `tanggal` | tanggal transaksi (teks `YYYY-MM-DD`) | temporal |
| `cabang` | nama cabang (penulisan tidak seragam) | kategorikal nominal |
| `kategori` | Kopi / Non-Kopi / Makanan | kategorikal nominal |
| `harga` | harga satuan, **teks**: `27000`, `Rp 27.000`, atau `?` | seharusnya numerik |
| `qty` | jumlah item | numerik diskret |
| `metode_bayar` | QRIS / Tunai / Kartu | kategorikal nominal |
| `umur_pelanggan` | umur pelanggan (ada NaN dan nilai mustahil) | numerik |
| `durasi_tunggu` | lama menunggu pesanan (menit) | numerik kontinu |
| `rating` | rating 1–5 (ada NaN) | ordinal |
| `pajak_persen` | persentase pajak | konstan |

### Setup dataset
Jalankan sel ini lebih dulu. **Jangan diubah.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.width', 120)


def buat_dataset(seed=8, n=640):
    """Data transaksi Kopi Senja (Sep-Okt 2026) yang sengaja dibuat kotor. JANGAN DIUBAH."""
    rng = np.random.default_rng(seed)
    tanggal = pd.Timestamp('2026-09-01') + pd.to_timedelta(rng.integers(0, 61, n), unit='D')
    cabang = rng.choice(['Bandung', 'Jakarta', 'Surabaya', 'Yogyakarta'], n, p=[.30, .34, .21, .15])
    kategori = rng.choice(['Kopi', 'Non-Kopi', 'Makanan'], n, p=[.50, .22, .28])
    dasar = pd.Series({'Kopi': 27000, 'Non-Kopi': 23000, 'Makanan': 33000})
    harga = dasar[kategori].to_numpy() + rng.integers(-4, 5, n) * 1000
    akhir_pekan = tanggal.dayofweek >= 5
    qty = rng.integers(1, 4, n) + akhir_pekan * rng.integers(0, 3, n)
    metode = rng.choice(['QRIS', 'Tunai', 'Kartu'], n, p=[.55, .25, .20])
    umur = np.clip(rng.normal(27, 7, n), 16, 65).round()
    durasi = (rng.gamma(2.2, 3.2, n) + np.where(cabang == 'Jakarta', 4, 0)).round(1)
    rating = np.clip(np.round(5.2 - 0.13 * durasi + rng.normal(0, .6, n)), 1, 5)

    df = pd.DataFrame({
        'id_transaksi': [f'TRX{i:04d}' for i in range(1, n + 1)],
        'tanggal': tanggal.strftime('%Y-%m-%d'), 'cabang': cabang, 'kategori': kategori,
        'harga': harga.astype(str), 'qty': qty, 'metode_bayar': metode,
        'umur_pelanggan': umur, 'durasi_tunggu': durasi, 'rating': rating, 'pajak_persen': 11,
    })
    # ---- membuat data kotor ----
    i = rng.choice(n, 70, replace=False)
    df.loc[i[:30], 'cabang'] = df.loc[i[:30], 'cabang'].str.lower()
    df.loc[i[30:50], 'cabang'] = df.loc[i[30:50], 'cabang'].str.upper()
    df.loc[i[50:], 'cabang'] = ' ' + df.loc[i[50:], 'cabang'] + ' '
    j = rng.choice(n, 160, replace=False)
    df.loc[j[:140], 'harga'] = ['Rp ' + f'{int(h):,}'.replace(',', '.') for h in df.loc[j[:140], 'harga']]
    df.loc[j[140:], 'harga'] = '?'
    k = rng.choice(n, 40, replace=False)
    df.loc[k[:6], 'umur_pelanggan'] = [999, 0, 150, 999, 2, 120]
    df.loc[k[6:], 'umur_pelanggan'] = np.nan
    df.loc[rng.choice(n, 45, replace=False), 'rating'] = np.nan
    dup = df.sample(18, random_state=seed)
    return pd.concat([df, dup]).sample(frac=1, random_state=seed).reset_index(drop=True)


df_raw = buat_dataset()
print(df_raw.shape)
print(df_raw.head(8).to_string())

---
## SOAL 1 — Pembersihan data (20 poin)

Implementasikan `bersihkanData(df_raw)` yang mengembalikan **DataFrame baru** yang sudah bersih (jangan ubah `df_raw`).
Lakukan langkah berikut **berurutan**:

1. **Seragamkan** `cabang`: hapus spasi di awal/akhir dan ubah ke *Title Case* (`" jakarta"` → `"Jakarta"`).
2. Ubah `harga` menjadi **numerik**: buang `"Rp"`, spasi, dan titik ribuan; nilai `"?"` menjadi `NaN`.
3. **Hapus baris duplikat**, lalu reset index.
4. Isi `harga` yang kosong dengan **median harga per kategori**.
5. `umur_pelanggan` di luar rentang **15–80** dianggap tidak valid → jadikan `NaN`, lalu isi semua `NaN` dengan **median**.
6. Isi `rating` yang kosong dengan **median**.
7. Ubah `tanggal` menjadi `datetime`.
8. **Buang kolom konstan** (hanya punya 1 nilai unik).
9. Tambahkan kolom `total = harga * qty` dan `akhir_pekan` (`True` jika Sabtu/Minggu).

*Implement bersihkanData(df_raw): standardize branch names, convert price text to numbers, drop duplicates, impute price by category median, invalidate ages outside 15–80 and impute with the median, impute rating, parse dates, drop constant columns, add total and akhir_pekan.*

In [ ]:
def bersihkanData(df_raw):
    df = df_raw.copy()
    # TODO: tulis langkah 1-9 di sini
    return df


# ---- KODE UJI (jangan diubah) ----
df = bersihkanData(df_raw)
print('Ukuran awal :', df_raw.shape)
print('Ukuran akhir:', df.shape)
print('Cabang      :', sorted(df['cabang'].unique()))
print('Missing     :', int(df.isna().sum().sum()))
print('Rentang umur:', df['umur_pelanggan'].min(), '-', df['umur_pelanggan'].max())
print(df.dtypes)

**Penjelasan:**

_tulis di sini_

---
## SOAL 2 — Penjualan per cabang (15 poin)

Manajer bertanya: **cabang mana yang penjualannya paling besar?**

1. Implementasikan `penjualanPerCabang(df)` yang mengembalikan DataFrame ber-index `cabang` dengan kolom
   `total_penjualan` (jumlah `total`), `jumlah_transaksi`, dan `rata_per_transaksi`, **terurut menurun** berdasarkan `total_penjualan`.
2. Buat **bar chart horizontal** total penjualan (dalam juta rupiah) dengan label nilai di ujung batang.

*Implement penjualanPerCabang(df) returning total sales, number of transactions and average per transaction by branch (sorted descending), then draw a labelled horizontal bar chart.*

In [ ]:
def penjualanPerCabang(df):
    # TODO
    pass


ringkas = penjualanPerCabang(df)
print(ringkas)

# TODO: bar chart horizontal total penjualan per cabang (juta Rp) + label nilai

**Penjelasan:**

_tulis di sini_

---
## SOAL 3 — Distribusi durasi tunggu (15 poin)

Pelanggan mengeluh soal antrean. Analisis **durasi tunggu**:

1. Implementasikan `statistikDurasi(df)` yang mengembalikan dictionary `{'mean', 'median', 'skew', 'n_outlier'}`
   (mean & median dibulatkan 2 desimal, skew 3 desimal, outlier memakai **aturan IQR 1,5×**).
2. Buat 2 panel berdampingan: **histogram** `durasi_tunggu` dengan garis vertikal mean & median, dan **boxplot** `durasi_tunggu` per cabang.

*Implement statistikDurasi(df) (mean, median, skewness, IQR outlier count) and plot a histogram with mean/median lines next to a boxplot by branch.*

In [ ]:
def statistikDurasi(df):
    # TODO
    pass


stat = statistikDurasi(df)
print(stat)

# TODO: subplot 1x2 -> histogram + garis mean/median | boxplot per cabang

**Penjelasan:**

_tulis di sini_

---
## SOAL 4 — Komposisi kategori produk (15 poin)

Apakah komposisi produk yang terjual **berbeda antar cabang**?

1. Implementasikan `komposisiKategori(df)` yang mengembalikan tabel **proporsi** transaksi tiap `kategori` per `cabang`
   (baris = cabang, kolom = kategori, **setiap baris berjumlah 1**, dibulatkan 3 desimal).
2. Visualisasikan dengan **100% stacked bar chart** horizontal, lengkap dengan label persentase di tiap segmen.

*Implement komposisiKategori(df) returning the share of each product category per branch (rows sum to 1), then draw a labelled 100% stacked bar chart.*

In [ ]:
def komposisiKategori(df):
    # TODO
    pass


komp = komposisiKategori(df)
print(komp)

# TODO: 100% stacked bar chart + label persentase

**Penjelasan:**

_tulis di sini_

---
## SOAL 5 — Tren penjualan harian (15 poin)

Manajer ingin melihat **tren penjualan harian** dan pengaruh **akhir pekan**.

1. Implementasikan `penjualanHarian(df)` yang mengembalikan **Series** total penjualan per tanggal. Index harus berupa
   tanggal **lengkap setiap hari** (hari tanpa transaksi bernilai 0).
2. Buat **line chart** penjualan harian (juta Rp) beserta **rata-rata bergerak 7 hari**, dan tandai titik akhir pekan.
3. Cetak rata-rata penjualan per hari untuk hari kerja vs akhir pekan.

*Implement penjualanHarian(df) returning a daily sales Series with every calendar day, plot it with a 7-day moving average and highlight weekends, then compare weekday vs weekend averages.*

In [ ]:
def penjualanHarian(df):
    # TODO
    pass


harian = penjualanHarian(df)
print(harian.describe().round(0))

# TODO: line chart harian + rolling 7 hari + tandai akhir pekan
# TODO: rata-rata penjualan per hari: hari kerja vs akhir pekan

**Penjelasan:**

_tulis di sini_

---
## SOAL 6 — Durasi tunggu vs rating (10 poin)

Apakah **menunggu lama membuat rating turun**?

1. Implementasikan `korelasiDurasiRating(df)` yang mengembalikan koefisien korelasi Pearson antara `durasi_tunggu` dan `rating` (3 desimal).
2. Buat 2 panel: **scatter plot** durasi vs rating dengan garis regresi (beri *jitter* karena rating berupa bilangan bulat), dan **heatmap korelasi** kolom numerik.

*Implement korelasiDurasiRating(df) (Pearson r, 3 decimals), then plot a jittered scatter with a regression line and a correlation heatmap of the numeric columns.*

In [ ]:
def korelasiDurasiRating(df):
    # TODO
    pass


r = korelasiDurasiRating(df)
print('r(durasi_tunggu, rating) =', r)

# TODO: scatter + garis regresi | heatmap korelasi

**Penjelasan:**

_tulis di sini_

---
## SOAL 7 — Perbaiki kode visualisasi (10 poin)

Kode di bawah ditulis mengikuti contoh lama dan mengandung **4 kesalahan** (error, peringatan *deprecated*, atau hasil yang menyesatkan).
Temukan dan perbaiki semuanya, lalu jelaskan tiap perbaikan.

*The code below contains 4 mistakes (errors, deprecated calls or misleading output). Find and fix them, then explain each fix.*

In [ ]:
# KODE YANG SALAH (untuk dibaca, jangan dijalankan apa adanya)
# fig, axs = plt.subplots(1, 3, figsize=(15, 4))
# sns.distplot(df['umur_pelanggan'], kde=False, bins=10, ax=axs[0])
# axs[0].set_title('Umur Pelanggan')
#
# axs[1].hist(df['durasi_tunggu'], bins=20, normed=True)
# axs[1].set_title('Durasi Tunggu')
#
# bayar = df['metode_bayar'].value_counts()
# warna = plt.get_cmap('Set2')(np.linspace(0, 1, 2))
# axs[2].pie(bayar, labels=bayar.index, autopct='%1.0f%%', colors=warna)
# axs[2].set_title('Metode Bayar')
# plt.show()
#
# print(df.corr())

In [ ]:
# TULIS KODE YANG SUDAH DIPERBAIKI DI SINI


**Penjelasan:**

_tulis di sini_

---
## Tugas Analisis
Jawab di sel teks di bawah setiap pertanyaan.

*Answer in the text cell below each question.*

### 1. Berdasarkan Soal 2, apakah cabang dengan total penjualan tertinggi juga yang nilai rata-rata per transaksinya tertinggi? Apa artinya?

*Based on Problem 2, does the branch with the highest total sales also have the highest average value per transaction? What does that imply?*

**Jawaban:** 

### 2. Mengapa nilai `durasi_tunggu` dan `umur_pelanggan` yang kosong sebaiknya diisi dengan median, bukan mean?

*Why should missing values be filled with the median rather than the mean?*

**Jawaban:** 

### 3. Manajer meminta 4 pie chart (satu per cabang) untuk komposisi kategori. Apakah itu pilihan terbaik? Jelaskan dengan konsep visual encoding.

*The manager asks for 4 pie charts (one per branch). Is that the best choice? Explain using visual-encoding concepts.*

**Jawaban:** 

### 4. Apa fungsi rata-rata bergerak 7 hari pada Soal 5, dan mengapa jendelanya 7?

*What is the purpose of the 7-day moving average in Problem 5, and why a window of 7?*

**Jawaban:** 

### 5. Berdasarkan Soal 3 dan Soal 6, rekomendasi apa yang kamu berikan ke manajer? Sertakan bukti dari grafik.

*Based on Problems 3 and 6, what would you recommend to the manager? Cite evidence from the charts.*

**Jawaban:** 

---
## Soal Teori

### 1. Apa beda visualisasi *exploratory* dan *explanatory*? Beri contoh dari studi kasus Kopi Senja.

**Jawaban:** 

### 2. Urutkan channel berikut dari yang paling akurat untuk data kuantitatif: *area, color, position, angle, length*.

**Jawaban:** 

### 3. Mengapa color hue (merah, hijau, biru) kurang tepat untuk data ordinal seperti rating 1–5? Apa alternatifnya?

**Jawaban:** 

### 4. Mengapa sumbu y bar chart wajib dimulai dari 0, sedangkan line chart tidak?

**Jawaban:** 

### 5. Pilih grafik yang paling tepat: (a) sebaran umur pelanggan; (b) tren penjualan 2 bulan; (c) hubungan harga dan qty; (d) perpindahan pelanggan dari kanal iklan ke cabang; (e) pangsa penjualan per cabang lalu per kategori.

**Jawaban:** 

### 6. Apa itu *redundant encoding*? Beri contoh.

**Jawaban:** 

### 7. Apa pelajaran dari *Anscombe's quartet*?

**Jawaban:** 